In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from nltk.corpus import wordnet as wn
import nltk
nltk.download('wordnet')


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [7]:

# Custom feature: WordNet-based semantic similarity
def semantic_features(X):
    features = []
    for pair in X:
        premise, hypothesis = pair.split(" [SEP] ")
        p_words = set(premise.split())
        h_words = set(hypothesis.split())

        # Synonym overlap
        syn_overlap = sum(1 for hw in h_words
                         if len(wn.synsets(hw)) > 0 and
                         any(wn.synsets(hw)[0].lowest_common_hypernyms(wn.synsets(pw)[0])
                             for pw in p_words if len(wn.synsets(pw)) > 0))

        # Antonym detection
        antonyms = sum(1 for hw in h_words for pw in p_words
                      if any(lem.antonyms()
                            for syn in wn.synsets(hw)
                            for lem in syn.lemmas()) and
                      hw != pw)

        features.append([syn_overlap, antonyms])
    return np.array(features)

# Load data
df = pd.read_csv("/content/train.csv")

# Split before feature extraction to prevent data leakage
X_raw = df[["premise", "hypothesis"]]
y = df["label"]
X_train_raw, X_dev_raw, y_train, y_dev = train_test_split(X_raw, y, test_size=0.2, stratify=y)


In [8]:

# Create pipeline with feature union
pipeline = Pipeline([
    ('features', FeatureUnion([
        ('tfidf', Pipeline([
            ('combine', FunctionTransformer(lambda x: x["premise"] + " [SEP] " + x["hypothesis"])),
            ('tfidf', TfidfVectorizer(
                lowercase=True,
                max_features=10000,
                ngram_range=(1, 3),
                analyzer='word',
                sublinear_tf=True
            ))
        ])),
        ('semantic', Pipeline([
            ('combine', FunctionTransformer(lambda x: x["premise"] + " [SEP] " + x["hypothesis"])),
            ('semantic_feats', FunctionTransformer(semantic_features))
        ]))
    ])),
    ('clf', SVC(
        kernel='linear',
        class_weight='balanced',  # Handle class imbalance
        C=0.1,
        probability=True
    ))
])


In [9]:

# Parameter grid for optimization
param_grid = {
    'clf__C': [0.01, 0.1, 1],
    'clf__kernel': ['linear', 'rbf'],
    'features__tfidf__tfidf__max_features': [5000, 10000]
}

# Grid search with 3-fold cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=3,
                          scoring='f1_macro', verbose=0)


In [10]:
grid_search.fit(X_train_raw, y_train)

KeyboardInterrupt: 

In [ ]:
# Best model
best_clf = grid_search.best_estimator_


In [ ]:

# Evaluate on dev set
y_pred = best_clf.predict(X_dev_raw)

# Enhanced evaluation metrics
print("Classification Report:\n", classification_report(y_dev, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_dev, y_pred))
print(f"Accuracy: {accuracy_score(y_dev, y_pred):.4f}")
print(f"Macro-F1: {f1_score(y_dev, y_pred, average='macro'):.4f}")
print(f"Macro-Precision: {precision_score(y_dev, y_pred, average='macro'):.4f}")
print(f"Macro-Recall: {recall_score(y_dev, y_pred, average='macro'):.4f}")
print(f"MCC: {matthews_corrcoef(y_dev, y_pred):.4f}")


In [ ]:

# Prepare test data
df_test = pd.read_csv("/content/dev.csv")
X_test_raw = df_test[["premise", "hypothesis"]]

# Predict on test data
test_preds = best_clf.predict(X_test_raw)


In [ ]:

# Save model and vectorizer for demo code
import joblib
joblib.dump(best_clf, 'nli_svm_model.joblib')

# Save predictions
output = pd.DataFrame({"prediction": test_preds})
output.to_csv("Group_52_A.csv", index=False)
